# 04. Gold Layer Processing

## Banking Transaction Lakehouse

**Purpose:** Transform validated transaction-level data into customer-level analytical metrics.

### Scope
- Prepare transaction-level contributions.
- Track updated and invalidated transactions.
- Calculate customer transaction metrics.
- Maintain incremental customer aggregations.
- Persist results in the Gold Delta table.

**Implementation:** Gold processing logic is maintained in `src/gold/gold_processor.py`.

In [0]:
from pyspark.sql import functions as F

from src.gold.gold_processor import process_gold

SILVER_TABLE = "workspace.default.demo_silver_transactions"
GOLD_TABLE = "workspace.default.demo_gold_customer_metrics"
CONTRIBUTION_TABLE = "workspace.default.demo_gold_transaction_contributions"
GOLD_CONTROL_TABLE = "workspace.default.demo_gold_batch_control"

batch_id = "bronze_demo_20261106_01"

print("Gold processing notebook initialized.")

In [0]:
# Create isolated Gold tables using existing development schemas

spark.sql("""
    CREATE TABLE IF NOT EXISTS workspace.default.demo_gold_customer_metrics
    LIKE workspace.default.dev_gold_customer_metrics
""")

spark.sql("""
    CREATE TABLE IF NOT EXISTS workspace.default.demo_gold_transaction_contributions
    LIKE workspace.default.dev_gold_transaction_contributions
""")

spark.sql("""
    CREATE TABLE IF NOT EXISTS workspace.default.demo_gold_batch_control
    LIKE workspace.default.dev_gold_batch_control
""")

# Read validated Silver transactions for this batch
silver_valid_batch = (
    spark.table(SILVER_TABLE)
    .filter(F.col("batch_id") == batch_id)
)

print("Valid Silver records ready for Gold:", silver_valid_batch.count())

display(silver_valid_batch)

In [0]:
# Process validated transactions into Gold
gold_result = process_gold(
    spark=spark,
    batch_id=batch_id,
    silver_valid_batch=silver_valid_batch,
    gold_table=GOLD_TABLE,
    gold_transaction_contribution_table=CONTRIBUTION_TABLE,
    gold_control_table=GOLD_CONTROL_TABLE
)

print("Gold processing completed.")

In [0]:
# Verify transaction-level contributions
contribution_verification = (
    spark.table(CONTRIBUTION_TABLE)
)

# Verify customer-level metrics
gold_verification = (
    spark.table(GOLD_TABLE)
)

print("Transaction contributions:")
display(contribution_verification)

print("Customer-level Gold metrics:")
display(gold_verification)

In [0]:
# Verify Gold batch processing status
gold_batch_status = (
    spark.table(GOLD_CONTROL_TABLE)
    .filter(F.col("batch_id") == batch_id)
)

display(gold_batch_status)